# EDA 01 — WESAD Dataset
Exploratory analysis of the WESAD wearable stress dataset.

Signals: EDA (4 Hz), BVP (64 Hz), TEMP (4 Hz), ACC (32 Hz)  
Labels: 0=baseline, 1=meditation, 2=stress (TSST), 3=amusement  
Subjects: S2–S17 (15 subjects, no S1/S12)

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

WESAD_ROOT = Path('/Users/omarnassar/Documents/Thesis/WESAD')
SUBJECTS = [d.name for d in sorted(WESAD_ROOT.iterdir()) if d.is_dir() and d.name.startswith('S')]

In [ ]:
# Load one subject to inspect structure
s2_path = WESAD_ROOT / 'S2' / 'S2.pkl'
with open(s2_path, 'rb') as f:
    s2 = pickle.load(f, encoding='latin1')

print('Top-level keys:', list(s2.keys()))
print('Signal keys:', list(s2['signal'].keys()))
print('Wrist keys:', list(s2['signal']['wrist'].keys()))
print('Label shape:', s2['label'].shape)
print('Label unique:', np.unique(s2['label']))

In [ ]:
# Label distribution across all subjects
label_map = {0: 'Undefined', 1: 'Baseline', 2: 'Stress (TSST)',
             3: 'Amusement', 4: 'Meditation', 5: 'Ignore 5', 6: 'Ignore 6', 7: 'Ignore 7'}

all_label_counts = {}
for s in SUBJECTS:
    pkl = WESAD_ROOT / s / f'{s}.pkl'
    if not pkl.exists():
        continue
    with open(pkl, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    for v in np.unique(data['label']):
        all_label_counts[v] = all_label_counts.get(v, 0) + int((data['label'] == v).sum())

fig, ax = plt.subplots(figsize=(8, 4))
labels_ordered = sorted(all_label_counts.keys())
counts = [all_label_counts[l] for l in labels_ordered]
ax.bar([label_map.get(l, str(l)) for l in labels_ordered], counts, color='steelblue', alpha=0.8)
ax.set_ylabel('Total samples (700 Hz chest reference)')
ax.set_title('WESAD: Label distribution across all subjects')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise raw signals for S2 (first 2000 samples of each wrist channel)
wrist = s2['signal']['wrist']
channels = {'EDA': wrist['EDA'].flatten(), 'BVP': wrist['BVP'].flatten(),
            'TEMP': wrist['TEMP'].flatten()}

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)
sample_rates = {'EDA': 4, 'BVP': 64, 'TEMP': 4}
n_sec = 60

for ax, (ch, sig) in zip(axes, channels.items()):
    fs = sample_rates[ch]
    n = fs * n_sec
    t = np.arange(n) / fs
    ax.plot(t, sig[:n], lw=0.8)
    ax.set_ylabel(ch)
    ax.set_xlabel('Time (s)')

fig.suptitle('S2 Wrist E4 — First 60 seconds')
plt.tight_layout()
plt.show()

In [ ]:
# EDA statistics per condition (S2)
eda = wrist['EDA'].flatten()   # 4 Hz
label = s2['label']             # 700 Hz

# Down-sample labels to 4 Hz (keep every 175th sample)
label_4hz = label[::175][:len(eda)]

# Only keep defined conditions 1-4
keep = np.isin(label_4hz, [1, 2, 3, 4])
df_eda = pd.DataFrame({'EDA': eda[keep], 'Condition': label_4hz[keep]})
df_eda['Condition'] = df_eda['Condition'].map(label_map)

fig, ax = plt.subplots(figsize=(7, 4))
df_eda.groupby('Condition')['EDA'].mean().plot(kind='bar', ax=ax, color='teal', alpha=0.8)
ax.set_ylabel('Mean EDA (μS)')
ax.set_title('S2: Mean EDA per condition')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Check processed windows shape
import numpy as np
processed_path = Path('../data/processed/wesad/S2_windows.npz')
if processed_path.exists():
    arr = np.load(processed_path)
    print(f'X shape: {arr["X"].shape}  (N windows, 4 channels, 1920 samples)')
    print(f'y shape: {arr["y"].shape}')
    print(f'Stress windows: {arr["y"].sum()} / {len(arr["y"])}')
else:
    print('Run script 01_preprocess_wesad.py first to generate processed windows.')